# 🚀 Step 4: Interactive Batch Prediction
Running inference and visualizing results.

In [1]:
import pandas as pd
import joblib
import plotly.express as px

# Load model and preprocessor
model = joblib.load('../models/model.joblib')
preprocessor = joblib.load('../data/processed/preprocessor.joblib')
data = pd.read_csv('../data/processed/credit_risk_cleaned.csv')

# Preprocess
X = data.drop('loan_status', axis=1)
X_processed = preprocessor.transform(X)

# FIX: Reconstruct exact feature names used during training (avoiding 'num__' prefixes)
numerical_cols = X.select_dtypes(exclude=['object']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)

# Combine names in the same order as fitted in Step 3
all_feature_names = numerical_cols + list(cat_feature_names)
X_processed_df = pd.DataFrame(X_processed, columns=all_feature_names)

# Predict
data['predicted_status'] = model.predict(X_processed_df)
data['default_prob'] = model.predict_proba(X_processed_df)[:, 1]

### 🔗 Leveraging Downstream Artifacts
This notebook is a perfect example of why we saved the `preprocessor.joblib` in the previous step:

*   **Consistency**: We are reusing the *exact* scaling and encoding parameters from the training set.
*   **Modular Pipeline**: Even if our raw data changes, we can rely on the preprocessor to "translate" it into the format the model expects.

> [!NOTE]
> In a production environment (like a FastAPI web server), this preprocessor is loaded once at startup and used for every incoming user request.


## 1. Prediction Visualization

In [2]:
fig = px.histogram(data, x='default_prob', color='predicted_status', 
                   title='Distribution of Default Probabilities', nbins=50)
fig.show()

### 📊 How to read the Probability Distribution
This chart shows the model's confidence for every person in the batch:

1.  **Certainty (The Peaks)**: High bars near **0.0** and **1.0** mean the model is very confident about those predictions.
2.  **Uncertainty (The Middle)**: Points around **0.5** are the "borderline" cases where the model is less sure.
3.  **The Threshold (0.5)**: By default, we use **0.5** as the cutoff. 
    *   **Blue (< 0.5)**: Predicted as Safe.
    *   **Red (> 0.5)**: Predicted as Default.

> [!TIP]
> **Business Strategy**: You can adjust this threshold! If you want to be extra cautious, you might set the cutoff at **0.3** to flag more people as "Risk," even if the model isn't 100% sure yet.


In [ ]:
data.to_csv('../data/processed/batch_results.csv', index=False)
print('Batch results saved!')
data.head()